In [9]:
from game import Connect4
import numpy as np
import math
import time
from tqdm.notebook import trange

import torch
print(torch.__version__)

import random

torch.manual_seed(0) #for reproducability

import torch.nn as nn
import torch.nn.functional as F

2.13.0


## The Model

In [10]:
class ResNet(nn.Module):
    def __init__(self, game, num_resBlocks, num_hidden, device):
        super().__init__()
        self.device = device
        self.startBlock = nn.Sequential(
            nn.Conv2d(3, num_hidden, kernel_size=3, padding=1), # 3 is the number of input planes that feed in
            nn.BatchNorm2d(num_hidden),
            nn.ReLU()
        )

        self.backBone = nn.ModuleList(
            [ResBlock(num_hidden) for _ in range(num_resBlocks)]
        )

        self.policyHead = nn.Sequential(
            nn.Conv2d(num_hidden, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Flatten(), # convulutional shape to linear nn shape
            nn.Linear(32 * game.row_count * game.col_count, game.col_count), #col count is the "action size"
        )

        self.valueHead = nn.Sequential(
            nn.Conv2d(num_hidden, 3, kernel_size=3, padding=1),
            nn.BatchNorm2d(3),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(3 * game.row_count * game.col_count,  1),
            nn.Tanh()
        )

        self.to(device)

    
    def forward(self, x):
        x = self.startBlock(x)
        
        for resBlock in self.backBone:
            x = resBlock(x)

        policy = self.policyHead(x)
        value = self.valueHead(x)

        return policy, value

        
class ResBlock(nn.Module):
    def __init__(self, num_hidden):
        super().__init__()
        self.conv1 = nn.Conv2d(num_hidden, num_hidden, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(num_hidden)
        self.conv2 = nn.Conv2d(num_hidden, num_hidden, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(num_hidden)

    def forward(self, x):
        residual = x
        
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))

        x = x + residual
        x = F.relu(x) #why F.relu and nn.relu?

        return x
    
     

## Test the Model

In [11]:
game = Connect4()
model = ResNet(game, 4, 64, 'cpu')
state = game.get_initial_state()

state = game.make_move(state, 1, 0)
state = game.make_move(state, -1, 3)
state = game.make_move(state, 1, 2)
state = game.make_move(state, -1, 3)

game.print_board(state)

encoded_state = game.get_encoded_state(state)
tensor_state = torch.tensor(encoded_state).unsqueeze(0)

policy, value = model(tensor_state)
value = value.item()
policy = torch.softmax(policy, axis = 1).squeeze(0).detach().cpu().numpy()

print(value)
print(policy)

  0 1 2 3 4 5 6
  ----------------
[[ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0 -1  0  0  0]
 [ 1  0  1 -1  0  0  0]]
  ----------------
0.1216319277882576
[0.2267328  0.23507373 0.10334568 0.0793746  0.1534288  0.09045508
 0.1115893 ]


In [12]:
def sample_move(valid_moves):
    chosen_move = np.random.choice(np.flatnonzero(valid_moves))
    #print(f"playing move {chosen_move}")
    return chosen_move

class Node:
    def __init__(self, args, game, state, player, action_taken=None, parent = None, prior=0) -> None:

        self.game = game
        self.args = args
        self.state = state
        self.player = player

        self.visit_count = 0
        self.win_count = 0
        self.parent = parent
        self.children = []
        self.action_taken = action_taken
        self.prior = prior #this is for the UCB function in the alpha alg


    def visit(self):
        self.visit_count += 1

    def is_terminal(self):
        if not self.action_taken:
            return False
        
        _, terminated = self.game.get_value_and_terminated(self.state, self.action_taken)
        return terminated

    def is_fully_expanded(self):
        return len(self.children) > 0

    def best_child(self): 
        
        if len(self.children) == 0:
            raise "no children to explore!"
        
        for child in self.children:
            if child.visit_count == 0:
                return child

        def ucb(child):
            if child.visit_count == 0:
                q_value = 0
            else:
                q_value = 1 - ((child.win_count / child.visit_count) + 1) / 2

            return  q_value + self.args['c'] * (math.sqrt(self.visit_count) / (child.visit_count + 1)) * child.prior 

        return max(self.children, key=ucb)
    
    def expand(self, policy):
        if self.is_fully_expanded():
            raise

        for action, prob in enumerate(policy):
            if prob > 0: # is this just for simplification? What would happen if we changed this to be slightly higher... interesting exp.
                child_state = self.game.make_move(self.state, self.player, action)
            
                child_node = Node(
                    args = self.args, 
                    game = self.game,
                    state = child_state,
                    player = self.game.get_opponent(self.player),
                    action_taken=action,
                    parent=self,
                    prior=prob
                )

                self.children.append(child_node)

    
    # def rollout(self): # dead code now that we use the model
        
    #     state = self.state
    #     action_taken = self.action_taken
    #     player = self.player

    #     while True:
    #         value, terminated = self.game.get_value_and_terminated(state, action_taken)
    #         if terminated:
    #             #print(f"game finished, returning {value * player * -1}")
    #             return value * player * -1

    #         valid_moves = self.game.get_valid_moves(state)
    #         action_taken = sample_move(valid_moves)

    #         state = self.game.make_move(state, player, action_taken)
    #         player = self.game.get_opponent(player)

    #         #self.game.print_board(state)


    def backpropogate(self, result):
        self.win_count += result
        self. ()

        if self.parent is not None:
            self.parent.backpropogate(-1 * result)
        

SyntaxError: invalid syntax (4126347077.py, line 98)

In [13]:
class MCTS:
    def __init__(self, game, args, model) -> None:
        self.game = game
        self.args = args
        self.model = model

    @torch.no_grad() #this means that torch does not compute the gradient map 
    def search(self, state, verbose = False):
       
        # define root
        root = Node(self.args, self.game, state, player=1)

        for i in range(self.args['num_searches']):
            current_node = root
            
            while not current_node.is_terminal() and current_node.is_fully_expanded():
                current_node = current_node.best_child()
                if verbose:
                    self.game.print_board(current_node.state)
            
            if not current_node.is_terminal():
                policy, value = self.model(
                    torch.tensor(self.game.get_encoded_state(current_node.state), device=self.model.device).unsqueeze(0)
                )
                policy = torch.softmax(policy, axis=1).squeeze(0).cpu().numpy()
                valid_moves = self.game.get_valid_moves(current_node.state)
                policy *= valid_moves
                policy /= np.sum(policy)

                value = value.item()

                current_node.expand(policy)
                    
            current_node.backpropogate(value)
            if verbose:
                print(f"iteration {i} finished, result = {result}")
        

        visits = np.zeros(self.game.col_count)
        for child in root.children:
            visits[child.action_taken] = child.visit_count

        visits /= np.sum(visits)
        return visits


In [14]:
class AlphaZero:
    def __init__(self, model, optimizer, game, args):
        self.model = model
        self.optimizer = optimizer
        self.game = game
        self.args = args
        self.mcts = MCTS(game, args, model)


    def selfPlay(self):
        memory = [] #tuple : (state, policy, player)
        player = 1
        state = self.game.get_initial_state()

        while True:
            #get neutral state ?
            if player == -1:
                neutral_state = self.game.change_perspective(state)
            else:
                neutral_state = state

            action_probs = self.mcts.search(neutral_state)
            memory.append((neutral_state, action_probs, player))

            action = np.random.choice(self.game.col_count, p=action_probs)
            state = self.game.make_move(state, player, action)

            value, is_terminal = self.game.get_value_and_terminated(state, action)
            if is_terminal:
                returnMemory = [] #tuple : (state, policy, result) result is from the perspective of the current player 

                for historical_state, historical_policy, historical_player in memory:
                    outcome = value if historical_player == player else -value
                    returnMemory.append((
                        self.game.get_encoded_state(historical_state), 
                        historical_policy, 
                        outcome
                    ))

                return returnMemory

            player = self.game.get_opponent(player)


    def train(self, memory):
        random.shuffle(memory)

        for batch_index in range(0, len(memory), self.args['batch_size']):
            sample = memory[batch_index: batch_index + self.args['batch_size']] #this seems wrong, should be moving over by batch_size

            state, policy_targets, value_targets = zip(*sample)
            state, policy_targets, value_targets = np.array(state), np.array(policy_targets), np.array(value_targets).reshape(-1, 1)

            state = torch.tensor(state, dtype=torch.float32, device=self.model.device)
            policy_targets = torch.tensor(policy_targets, dtype=torch.float32, device=self.model.device)
            value_targets = torch.tensor(value_targets, dtype=torch.float32, device=self.model.device)

            out_policy, out_value = self.model(state)

            policy_loss = F.cross_entropy(out_policy, policy_targets)
            value_loss = F.mse_loss(out_value, value_targets)

            loss = policy_loss + value_loss #why can we just add them together like this

            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

 
    def learn(self):
        for iteration in range(self.args['num_iterations']):
            memory =[]

            self.model.eval()
            for selfPlay_iteration in trange(self.args['num_selfPlay_iterations']):
                memory.extend(self.selfPlay())

            self.model.train()
            for epoch in trange(self.args['num_epochs']):
                self.train(memory)

            torch.save(self.model.state_dict(), f"model_{iteration}.pt")
            torch.save(self.optimizer.state_dict(), f"optimizer_{iteration}.pt")

## Training Cell 

(takes ~5 mins at these settings)

In [15]:
game = Connect4()
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

model = ResNet(game, 4, 64, device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0001) #weight_decay is L2 regularization

args = {
    "c" : 2,
    "num_searches" : 150,
    "num_iterations" : 6,
    "num_selfPlay_iterations" : 250,
    "num_epochs" : 4,
    "batch_size" : 64
}

alpha = AlphaZero(model, optimizer, game, args)
alpha.learn()

  0%|          | 0/250 [00:00<?, ?it/s]

/var/folders/vl/_6prr5hj1wj73zp5dg5w1bcm0000gn/T/ipykernel_80840/107347716.py:28: RuntimeWarning: invalid value encountered in divide
  policy /= np.sum(policy)


  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/250 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

...

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/250 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/250 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

## Use the trained model!

In [19]:
connect4 = Connect4()
state = connect4.get_initial_state()
connect4.print_board(state)

model = ResNet(game, 4, 64, device)
model.load_state_dict(torch.load('model_5.pt'))
model.eval()

player = 1 
while True:
    if player == -1:
        legal_moves = connect4.get_valid_moves(state)
        print(f"Legal Moves: {[i for i in range(connect4.col_count) if legal_moves[i]]}")
        
        action = int(input(f"player {player}: "))
        if action < 0 or action >= connect4.col_count or not legal_moves[action]:
            print('illegal move')
            continue

    else:
        policy, value = model(torch.Tensor(connect4.get_encoded_state(state)).unsqueeze(0).to(device))
        probs = torch.softmax(policy, axis=1).squeeze(0).detach().cpu().numpy()
        value = value.item()

        print(probs)
        print(value)

        action = np.argmax(probs)
        print(f"AI is playing column {action}")

    state = connect4.make_move(state, player, action)
    connect4.print_board(state)
    value, terminated = connect4.get_value_and_terminated(state, action)

    if terminated:
        if value == 1:
            print(f"Player {player} wins!")
        else:
            print("draw")
        break

    player = connect4.get_opponent(player)
    


  0 1 2 3 4 5 6
  ----------------
[[0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0]]
  ----------------
[0.1761525  0.17474633 0.15873764 0.16782005 0.08990293 0.10141446
 0.13122615]
0.08001969009637833
AI is playing column 0
  0 1 2 3 4 5 6
  ----------------
[[0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0]
 [1 0 0 0 0 0 0]]
  ----------------
Legal Moves: [0, 1, 2, 3, 4, 5, 6]
  0 1 2 3 4 5 6
  ----------------
[[ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 1  0  0 -1  0  0  0]]
  ----------------
[0.16860038 0.3505648  0.08651555 0.07267014 0.08221114 0.21200547
 0.02743243]
-0.4279988706111908
AI is playing column 1
  0 1 2 3 4 5 6
  ----------------
[[ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 1  1  0 -1  0  0  0]]
  ----------------
Legal M

In [16]:
game = Connect4()
model = ResNet(game, 4, 64)
model.eval() #what does this do?

state_init = game.get_initial_state()

round_1 = MCTS(game=game, args={'iterations' : 2000, 'c' : 2}, model=model)
results = round_1.search(state=state_init, verbose=False)

In [17]:
results

array([300., 289., 290., 303., 269., 271., 277.])

In [20]:
connect4 = Connect4()
state = connect4.get_initial_state()
model = ResNet(connect4, 4, 64)

mcts = MCTS(game=connect4, args={'iterations' : 8000, 'c' : 2}, model=model)
model.eval()

player = 1 
while True:
    if player == -1:
        legal_moves = connect4.get_valid_moves(state)
        print(f"Legal Moves: {[i for i in range(connect4.col_count) if legal_moves[i]]}")
        
        action = int(input(f"player {player}: "))
        if action < 0 or action >= connect4.col_count or not legal_moves[action]:
            print('illegal move')
            continue

    else:
        search = mcts.search(state=state)
        action = np.argmax(search)

    state = connect4.make_move(state, player, action)
    print(f"player {player} plays {action}")
    connect4.print_board(state)

    value, terminated = connect4.get_value_and_terminated(state, action)

    if terminated:
        if value == 1:
            print(f"Player {player} wins!")
        else:
            print("draw")
        break


    player = connect4.get_opponent(player)


player 1 plays 4
  0 1 2 3 4 5 6
  ----------------
[[0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0]]
  ----------------
Legal Moves: [0, 1, 2, 3, 4, 5, 6]
player -1 plays 2
  0 1 2 3 4 5 6
  ----------------
[[ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0 -1  0  1  0  0]]
  ----------------
player 1 plays 4
  0 1 2 3 4 5 6
  ----------------
[[ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  1  0  0]
 [ 0  0 -1  0  1  0  0]]
  ----------------
Legal Moves: [0, 1, 2, 3, 4, 5, 6]
player -1 plays 1
  0 1 2 3 4 5 6
  ----------------
[[ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  1  0  0]
 [ 0 -1 -1  0  1  0  0]]
  ----------------
player 1 plays 2
  0 1 2 3 4 5 6
  ----------------
[[ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  